In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
save_dir = "/content/drive/MyDrive/NLP_Project_Preprocessing"

In [ ]:
pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 89.3 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pickle
from gensim.models import Word2Vec

In [ ]:
X_train_pad = np.load(f"{save_dir}/X_train_pad.npy")
X_test_pad = np.load(f"{save_dir}/X_test_pad.npy")

y_train_stance = np.load(f"{save_dir}/y_train_stance.npy")
y_test_stance = np.load(f"{save_dir}/y_test_stance.npy")

In [ ]:
embedding_matrix = np.load(
    f"{save_dir}/embedding_matrix.npy"
)

In [ ]:
w2v_model = Word2Vec.load(
    f"{save_dir}/word2vec.model"
)

In [ ]:
with open(f"{save_dir}/word_index.pkl", "rb") as f:
    word_index = pickle.load(f)

vocab_size = len(word_index) + 2

In [ ]:
import joblib

stance_encoder = joblib.load(
    f"{save_dir}/stance_encoder.pkl"
)

In [ ]:
print("X_train_pad:", X_train_pad.shape)
print("X_test_pad:", X_test_pad.shape)

print("y_train_stance:", y_train_stance.shape)
print("y_test_stance:", y_test_stance.shape)

print("Embedding Matrix:", embedding_matrix.shape)

print("Vocabulary Size:", len(word_index))

X_train_pad: (1166475, 30)
X_test_pad: (291619, 30)
y_train_stance: (1166475,)
y_test_stance: (291619,)
Embedding Matrix: (81958, 100)
Vocabulary Size: 81956


##model 1 computed class wts

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train_stance)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_stance
)

class_weights = dict(zip(classes, weights))
print(class_weights)

{np.int64(0): np.float64(19.825871915153986), np.int64(1): np.float64(4.280186695727794), np.int64(2): np.float64(0.3681985189674438)}


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout

rnn_model = Sequential([
    Embedding(
        input_dim=embedding_matrix.shape[0],
        output_dim=embedding_matrix.shape[1],
        weights=[embedding_matrix],
        trainable=False
    ),

    SimpleRNN(64),

    Dropout(0.5),

    Dense(32, activation="relu"),

    Dense(3, activation="softmax")
])

In [ ]:
rnn_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

In [ ]:
history = rnn_model.fit(
    X_train_pad,
    y_train_stance,
    validation_split=0.1,
    epochs=10,
    batch_size=128,
    class_weight=class_weights,
    callbacks=[early_stop]
)

Epoch 1/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 53s 6ms/step - accuracy: 0.3997 - loss: 1.0771 - val_accuracy: 0.0177 - val_loss: 1.1097
Epoch 2/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 45s 5ms/step - accuracy: 0.3784 - loss: 1.0926 - val_accuracy: 0.0172 - val_loss: 1.1033
Epoch 3/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 45s 5ms/step - accuracy: 0.3614 - loss: 1.0984 - val_accuracy: 0.0172 - val_loss: 1.1038
Epoch 4/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 64s 8ms/step - accuracy: 0.3660 - loss: 1.0983 - val_accuracy: 0.0763 - val_loss: 1.0958
Epoch 5/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 44s 5ms/step - accuracy: 0.4564 - loss: 1.0980 - val_accuracy: 0.9057 - val_loss: 1.0679
Epoch 6/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 47s 6ms/step - accuracy: 0.4549 - loss: 1.0985 - val_accuracy: 0.0173 - val_loss: 1.0992
Epoch 7/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 45s 5ms/step - accuracy: 0.5568 - loss: 1.0874 - val_accuracy: 0.6777 - val_loss: 1.0411
Epoch 8/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 44s 5ms/step - accuracy: 0.4652 - loss: 1

In [ ]:
test_loss, test_acc = rnn_model.evaluate(
    X_test_pad,
    y_test_stance,
    verbose=1
)

print("Test Accuracy:", test_acc)

9114/9114 ━━━━━━━━━━━━━━━━━━━━ 30s 3ms/step - accuracy: 0.6798 - loss: 1.0403
Test Accuracy: 0.6797910928726196


In [ ]:
import numpy as np

y_pred_probs = rnn_model.predict(X_test_pad)

y_pred = np.argmax(y_pred_probs, axis=1)

9114/9114 ━━━━━━━━━━━━━━━━━━━━ 21s 2ms/step


In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test_stance,
        y_pred,
        target_names=[
            "Pro Russia",
            "Pro Ukraine",
            "Unsure"
        ]
    )
)

              precision    recall  f1-score   support

  Pro Russia       0.03      0.46      0.06      4902
 Pro Ukraine       0.08      0.05      0.07     22711
      Unsure       0.92      0.74      0.82    264006

    accuracy                           0.68    291619
   macro avg       0.35      0.42      0.32    291619
weighted avg       0.84      0.68      0.75    291619



In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_stance, y_pred)

print(cm)

[[  2245    322   2335]
 [  7204   1225  14282]
 [ 55889  13347 194770]]


#model 2 class wts + trainable embeddings

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train_stance)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_stance
)

class_weights = dict(zip(classes, weights))
print(class_weights)

{np.int64(0): np.float64(19.825871915153986), np.int64(1): np.float64(4.280186695727794), np.int64(2): np.float64(0.3681985189674438)}


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout

rnn_model_2 = Sequential([
    Embedding(
        input_dim=embedding_matrix.shape[0],
        output_dim=embedding_matrix.shape[1],
        weights=[embedding_matrix],
        trainable=True
    ),

    SimpleRNN(64),

    Dropout(0.5),

    Dense(32, activation="relu"),

    Dense(3, activation="softmax")
])

In [ ]:
rnn_model_2.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history = rnn_model_2.fit(
    X_train_pad,
    y_train_stance,
    validation_split=0.1,
    epochs=10,
    batch_size=128,
    class_weight=class_weights,
    callbacks=[early_stop]
)

Epoch 1/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 62s 7ms/step - accuracy: 0.5132 - loss: 1.0292 - val_accuracy: 0.6338 - val_loss: 0.9273
Epoch 2/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 53s 6ms/step - accuracy: 0.4783 - loss: 1.0813 - val_accuracy: 0.5769 - val_loss: 1.0060
Epoch 3/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 53s 7ms/step - accuracy: 0.4914 - loss: 1.0751 - val_accuracy: 0.6919 - val_loss: 0.9704


In [ ]:
test_loss, test_acc = rnn_model_2.evaluate(
    X_test_pad,
    y_test_stance,
    verbose=1
)

print("Test Accuracy:", test_acc)

9114/9114 ━━━━━━━━━━━━━━━━━━━━ 37s 4ms/step - accuracy: 0.6350 - loss: 0.9256
Test Accuracy: 0.6350409388542175


In [ ]:
import numpy as np

y_pred_probs = rnn_model_2.predict(X_test_pad)

y_pred = np.argmax(y_pred_probs, axis=1)

9114/9114 ━━━━━━━━━━━━━━━━━━━━ 20s 2ms/step


In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test_stance,
        y_pred,
        target_names=[
            "Pro Russia",
            "Pro Ukraine",
            "Unsure"
        ]
    )
)

              precision    recall  f1-score   support

  Pro Russia       0.04      0.37      0.06      4902
 Pro Ukraine       0.20      0.54      0.29     22711
      Unsure       0.96      0.65      0.77    264006

    accuracy                           0.64    291619
   macro avg       0.40      0.52      0.38    291619
weighted avg       0.88      0.64      0.72    291619



In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_stance, y_pred)
print(cm)

[[  1791   1418   1693]
 [  4434  12316   5961]
 [ 44314  48609 171083]]


In [ ]:
rnn_model_2.save(f"{save_dir}/rnn_stance.keras")

In [ ]:
import os

print(os.listdir(save_dir))

['sentiment_encoder.pkl', 'stance_encoder.pkl', 'train_idx.npy', 'test_idx.npy', 'y_train_stance.npy', 'y_test_stance.npy', 'y_train_sentiment.npy', 'y_test_sentiment.npy', 'X_train.csv', 'X_test.csv', 'X_train_tokens.pkl', 'X_test_tokens.pkl', 'word_index.pkl', 'word2vec.model', 'X_train_pad.npy', 'X_test_pad.npy', 'embedding_matrix.npy', 'rnn_stance.keras']


##model 3 with wts= 10,3,1 and trainable=true

In [ ]:
class_weights = {
    0: 10,
    1: 3,
    2: 1
}

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout

rnn_model_3 = Sequential([
    Embedding(
        input_dim=embedding_matrix.shape[0],
        output_dim=embedding_matrix.shape[1],
        weights=[embedding_matrix],
        trainable=True
    ),

    SimpleRNN(64),

    Dropout(0.5),

    Dense(32, activation="relu"),

    Dense(3, activation="softmax")
])

In [ ]:
rnn_model_3.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history = rnn_model_3.fit(
    X_train_pad,
    y_train_stance,
    validation_split=0.1,
    epochs=10,
    batch_size=128,
    class_weight=class_weights,
    callbacks=[early_stop]
)

Epoch 1/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 73s 8ms/step - accuracy: 0.8927 - loss: 1.0061 - val_accuracy: 0.8754 - val_loss: 0.5663
Epoch 2/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 69s 7ms/step - accuracy: 0.8995 - loss: 0.9851 - val_accuracy: 0.9057 - val_loss: 0.4605
Epoch 3/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 82s 7ms/step - accuracy: 0.8887 - loss: 0.9602 - val_accuracy: 0.8828 - val_loss: 0.4638
Epoch 4/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 55s 7ms/step - accuracy: 0.8865 - loss: 0.9715 - val_accuracy: 0.9057 - val_loss: 0.4963


In [ ]:
test_loss, test_acc = rnn_model_3.evaluate(
    X_test_pad,
    y_test_stance,
    verbose=1
)

print("Test Accuracy:", test_acc)

9114/9114 ━━━━━━━━━━━━━━━━━━━━ 30s 3ms/step - accuracy: 0.9053 - loss: 0.4599
Test Accuracy: 0.9052736759185791


In [ ]:
import numpy as np

y_pred_probs = rnn_model_3.predict(X_test_pad)

y_pred = np.argmax(y_pred_probs, axis=1)

9114/9114 ━━━━━━━━━━━━━━━━━━━━ 20s 2ms/step


In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test_stance,
        y_pred,
        target_names=[
            "Pro Russia",
            "Pro Ukraine",
            "Unsure"
        ]
    )
)

              precision    recall  f1-score   support

  Pro Russia       0.00      0.00      0.00      4902
 Pro Ukraine       0.00      0.00      0.00     22711
      Unsure       0.91      1.00      0.95    264006

    accuracy                           0.91    291619
   macro avg       0.30      0.33      0.32    291619
weighted avg       0.82      0.91      0.86    291619



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_stance, y_pred)

print(cm)

[[     0      0   4902]
 [     0      0  22711]
 [     0     11 263995]]


##model 4 (15,4,1) + trainable = true

In [ ]:
class_weights = {
    0: 15,
    1: 4,
    2: 1
}

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout

rnn_model_4 = Sequential([
    Embedding(
        input_dim=embedding_matrix.shape[0],
        output_dim=embedding_matrix.shape[1],
        weights=[embedding_matrix],
        trainable=True
    ),

    SimpleRNN(64),

    Dropout(0.5),

    Dense(32, activation="relu"),

    Dense(3, activation="softmax")
])

In [ ]:
rnn_model_4.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history = rnn_model_4.fit(
    X_train_pad,
    y_train_stance,
    validation_split=0.1,
    epochs=10,
    batch_size=128,
    class_weight=class_weights,
    callbacks=[early_stop]
)

Epoch 1/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 69s 8ms/step - accuracy: 0.8667 - loss: 1.2778 - val_accuracy: 0.9057 - val_loss: 0.6113
Epoch 2/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 69s 7ms/step - accuracy: 0.9053 - loss: 1.3671 - val_accuracy: 0.9057 - val_loss: 0.6276


In [ ]:
test_loss, test_acc = rnn_model_4.evaluate(
    X_test_pad,
    y_test_stance,
    verbose=1
)

print("Test Accuracy:", test_acc)

9114/9114 ━━━━━━━━━━━━━━━━━━━━ 32s 3ms/step - accuracy: 0.9053 - loss: 0.6116
Test Accuracy: 0.9053114056587219


In [ ]:
import numpy as np

y_pred_probs = rnn_model_4.predict(X_test_pad)

y_pred = np.argmax(y_pred_probs, axis=1)

9114/9114 ━━━━━━━━━━━━━━━━━━━━ 20s 2ms/step


In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test_stance,
        y_pred,
        target_names=[
            "Pro Russia",
            "Pro Ukraine",
            "Unsure"
        ]
    )
)

              precision    recall  f1-score   support

  Pro Russia       0.00      0.00      0.00      4902
 Pro Ukraine       0.00      0.00      0.00     22711
      Unsure       0.91      1.00      0.95    264006

    accuracy                           0.91    291619
   macro avg       0.30      0.33      0.32    291619
weighted avg       0.82      0.91      0.86    291619



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_stance, y_pred)

print(cm)

[[     0      0   4902]
 [     0      0  22711]
 [     0      0 264006]]
